In [1]:
import pandas as pd
from konlpy.tag import Okt
okt = Okt()

In [2]:
text = "안녕하세요. 저는 홍길동 입니다. 팀에서 도적을 담당하고 있습니다."

In [3]:
# 형태소 단위로 분리
okt.morphs(text)

['안녕하세요',
 '.',
 '저',
 '는',
 '홍길동',
 '입니다',
 '.',
 '팀',
 '에서',
 '도적',
 '을',
 '담당',
 '하고',
 '있습니다',
 '.']

In [4]:
# 명사만
okt.nouns(text)

['저', '홍길동', '팀', '도적', '담당']

In [5]:
# 어절 단위
okt.phrases(text)

['홍길동', '도적', '담당']

In [6]:
okt.pos(text)

[('안녕하세요', 'Adjective'),
 ('.', 'Punctuation'),
 ('저', 'Noun'),
 ('는', 'Josa'),
 ('홍길동', 'Noun'),
 ('입니다', 'Adjective'),
 ('.', 'Punctuation'),
 ('팀', 'Noun'),
 ('에서', 'Josa'),
 ('도적', 'Noun'),
 ('을', 'Josa'),
 ('담당', 'Noun'),
 ('하고', 'Josa'),
 ('있습니다', 'Adjective'),
 ('.', 'Punctuation')]

In [7]:
train_df = pd.read_table("data/ratings_train.txt") # csv는 아닌데 볼 때 테이블처럼 생겼다 싶으면 read_table 써보시게
test_df = pd.read_table("data/ratings_test.txt")

In [8]:
train_df = train_df.fillna(" ")
test_df = test_df.fillna(" ")

In [9]:
train_df.info(), test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  150000 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  50000 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


(None, None)

In [10]:
# 임베딩
okt = Okt()

def tw_tokenzier(text):
    tokenzier_ko = okt.morphs(text)
    return tokenzier_ko

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer(tokenizer=tw_tokenzier, ngram_range=(1,2), min_df=3, max_df=0.9) # min_df=3 : 3회
tfidf_vect.fit(train_df["document"])
tfidf_matrix_train = tfidf_vect.transform(train_df["document"])

C:\Users\user\AppData\Local\Programs\Python\Python39\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [12]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=3.5, random_state=42)
lr.fit(tfidf_matrix_train, train_df["label"]) # matrix(선형OO) 데이터인 희소행렬을 선형알고리즘으로 70~80% 나오면(결과값이 잘 나오면) 앙상블 써야지 말고 선형으로 높이는 노력해봐라 -> grid search

LogisticRegression(C=3.5, random_state=42)

In [13]:
tfidf_matrix_test = tfidf_vect.transform(test_df["document"])
preds = lr.predict(tfidf_matrix_test)

In [14]:
from sklearn.metrics import accuracy_score
accuracy_score(test_df["label"], preds)

0.86532

In [ ]:
import joblib

joblib.dump(lr, "lr_v1.pkl")
joblib.dump(tfidf_vect, "tfidf_vect.pkl") # NLP는 fitting 시킨 것(전처리)도 저장 // 최대한 빨리 하려고
# 숫자로 전처리 많이하고 민맥스, 로버스터 다 쓰면 그것도 덤프 뜨기

['tfidf_vect.pkl']